Synthetic Geodesics

Geodesic results on the linear / bifurcation / tree Splatter benchmarks.

Paths are refined with the non-amortised graph-free procedure (straight chord $\rightarrow$
Adam on the discrete pullback energy under the ambient extended feature map $f^{+}=[P;\beta s]$,
$\beta=10$, 31 waypoints, 1000 steps) and then projected; lengths are estimated on the
projected polyline. The ground truth is the exact arc length on the clean `BaseCellMeans`
tree (`context/SYNTHETIC_GEODESICS.md`).

## What is measured

| § | Question | Metric |
|---|---|---|
| 1 | Is the refined projected length closer to the tree ground truth than the standard graph estimator? | per-pair MSE $(L_{\mathrm{method}} - L_{\mathrm{tree}})^{2}$ |
| 2 | Which estimator stays on the ground-truth manifold? | per-pair mean distance of the interior waypoints to the clean-tree polyline |
| 3 | The same two figures for the full difficulty ladder (linear → bifurcation → tree) in one pass | — |
| 4 | Summary table for the paper (scenarios × methods, mean $\pm$ std over seeds, bold best, LaTeX output) | — |
| 5 | Combined paper figure: 2 rows (metrics) × 3 columns (scenarios), shared $y$ per row | — |

Both figures use the **4-seed sweep** `geodesics-full-s42..s45`
(4 seeds $\times$ 2000 pairs per scenario): the box pools the per-pair distribution over
seeds, and the white shapes mark each seed's **mean** — the aggregate reported in
`results/synthetic_data/analysis/geodesics_comparison.md`: $\circ$ s42, $\square$ s43, $\triangle$ s44, $\diamond$ s45.
`graph_noisy_k15` is the $k$NN-15 Dijkstra distance on the noisy PCA-50 cloud, i.e. the
standard graph-based estimator.

All saved figures are **≥ 300 DPI** (SVG + PNG via `experiments.paper_figures.style.savefig`).

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from IPython.display import Image, Markdown, display


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / "experiments" / "toy_manifolds").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root (expected experiments/toy_manifolds/)."
    )


REPO_ROOT = _find_repo_root()

for _p in (REPO_ROOT, REPO_ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from notebooks.helpers import sc_geodesics as geo
from notebooks.helpers.style import DPI, apply_paper_style


def show_fig(path: Path) -> None:
    """Display the PNG sibling inline (savefig writes both PNG and SVG)."""
    print("saved:", path)
    display(Image(filename=str(path.with_suffix(".png"))))


apply_paper_style()
print(f"REPO_ROOT  = {REPO_ROOT}")
print(f"Figure DPI = {DPI}")


In [ ]:
RESULTS_ROOT_NAME = "synthetic_data"  # "synthetic_data" for the paper runs
RESULTS_ROOT = REPO_ROOT / "results" / RESULTS_ROOT_NAME
OUT_DIR = REPO_ROOT / "notebooks" / "figures" / "sc_geodesics"

SEED_PATTERN = "geodesics-full-s*"
SCENARIOS = ["tree"]
ALL_SCENARIOS = ["linear", "bifurcation", "tree"]
METHODS = ("refine_proj", "graph_noisy_k15")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results root: {RESULTS_ROOT}")
print(f"Saving figures under: {OUT_DIR}\n")

RUNS: dict[str, list[tuple[str, str]]] = {}
for scenario in SCENARIOS:
    RUNS[scenario] = geo.find_geodesics_runs(
        RESULTS_ROOT, scenario, pattern=SEED_PATTERN
    )
    seeds = ", ".join(label for _, label in RUNS[scenario]) or "none"
    print(f"  {scenario:12s} {len(RUNS[scenario])} runs ({seeds})")


## Summary table

In [ ]:
if not any(
    geo.find_geodesics_runs(RESULTS_ROOT, s, pattern=SEED_PATTERN)
    for s in ALL_SCENARIOS
):
    print("No Stage III geodesics runs yet; skipping the summary table.")
else:
    table_rows = geo.collect_seed_metrics(
        RESULTS_ROOT, ALL_SCENARIOS, pattern=SEED_PATTERN, methods=METHODS
    )
    display(Markdown(geo.format_seed_table_markdown(table_rows)))
    print(geo.format_seed_table_latex(table_rows))


## 5. Summary figure

In [ ]:
if not any(
    geo.find_geodesics_runs(RESULTS_ROOT, s, pattern=SEED_PATTERN)
    for s in ALL_SCENARIOS
):
    print("No Stage III geodesics runs yet; skipping the overview figure.")
else:
    path, info = geo.fig_metrics_overview(
        OUT_DIR, RESULTS_ROOT, ALL_SCENARIOS, pattern=SEED_PATTERN, methods=METHODS
    )
    show_fig(path)
    for scenario in ALL_SCENARIOS:
        mse = info["panels"][f"mse:{scenario}"]["pooled_mean"]
        dist = info["panels"][f"offgt:{scenario}"]["pooled_mean"]
        print(
            f"  {scenario:12s} "
            + " | ".join(f"MSE {m}: {v:.3g}" for m, v in mse.items())
            + " | "
            + " | ".join(f"dist {m}: {v:.3g}" for m, v in dist.items())
        )
